# Beyond Majority Voting: Selecting LLM Answers via Hidden State Trajectory Probes

**Author:** Nikolay Yudin (`n.yudin@gmail.com`)
**Repository:** [github.com/nick-yudin/Generalization/.../Latent\_control](https://github.com/nick-yudin/Generalization/tree/main/papers/Latent_control)

## Abstract

When a language model generates multiple candidate answers, how should we pick the best one? The default strategy — majority voting — treats the model as a black box, discarding everything except final answer strings. We show that the model's internal computations already contain a usable signal for answer quality, and that a remarkably simple method can extract it.

We propose *trajectory probes*: lightweight linear models trained on hidden-state features aggregated across the generation process. From each candidate answer, we extract mean, standard deviation, and final-token activations at eight evenly spaced layers, projected to 256 dimensions — a 6,144-dimensional trajectory fingerprint. A logistic regression probe trained with a pairwise ranking objective (RankNet) learns to prefer correct answers over incorrect ones from the same question.

On TriviaQA (Llama-3.1-8B-Instruct, *t*=0.3, *n*=500, *K*=4, 3 seeds), the probe improves over majority voting by **+5.1 ± 0.1 pp** (56.4% vs 51.3%), recovering 58% of the gap to the oracle upper bound, with a selection precision (PickAcc) of 91.2 ± 1.7%. On MATH (*n*=500, *K*\_gen=6, 3 seeds), the relationship is *K*-dependent: the probe outperforms MV by +2.1 pp at *K*\_eval=2 (all seeds positive) but the advantage narrows to +0.6 pp at *K*\_eval=4, as MV benefits more from additional votes in mathematical reasoning. The probe trains in under 60 seconds on CPU from 500 examples and adds zero latency at inference.

Two findings surprised us. First, the choice of training objective matters more than feature quality: a binary classifier with higher cross-validated AUC can underperform a pairwise probe with lower AUC, because ranking among candidates is a fundamentally different task than classifying correctness in isolation. Second, the per-layer signal distribution acts as a domain fingerprint — factual recall spreads information across all layers while mathematical reasoning concentrates it in the final third — yet a single probe trained on mixed-domain data matches domain-specific specialists with no interference.

Our results suggest that the "verifier" for best-of-*K* selection need not be a separate model or an additional LLM call. It can be a linear function of what the model already computes.

---
## Notebook 01: End-to-End Pipeline (TriviaQA & MATH)

**Runtime:** GPU required. Sanity mode ~5 min, canonical mode ~30 min (TriviaQA) / ~6 hrs (MATH).
**Model:** `meta-llama/Meta-Llama-3.1-8B-Instruct` (requires HuggingFace access token)

### How to use
1. Set `DOMAIN`, `MODE`, and `SEED` in the config cell below
2. Run all cells
3. Compare output with expected values

### Expected outputs

| Setting | Probe | MV | Δ | PickAcc |
|---------|-------|----|---|---------|
| `trivia`, `canon`, seed=0 | ~60.8% | ~55.8% | ~+5.0pp | ~93% |
| `trivia`, `canon`, seed=1 | ~53.6% | ~48.4% | ~+5.2pp | ~90% |
| `trivia`, `canon`, seed=2 | ~54.8% | ~49.8% | ~+5.0pp | ~91% |
| `math`, `canon`, seed=0 | ~49.6% | ~48.4% | ~+1.2pp | ~85% |
| `math`, `canon`, seed=1 | ~49.0% | ~47.8% | ~+1.2pp | ~83% |
| `math`, `canon`, seed=2 | ~46.8% | ~47.4% | ~-0.6pp | ~82% |
| `trivia`, `sanity`, any seed | Pipeline completes, numbers approximate | | | |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — change these and run all cells below       ║
# ╚══════════════════════════════════════════════════════════════╝

DOMAIN = "trivia"   # "trivia" or "math"
MODE   = "sanity"   # "sanity" (n=50, ~5 min) or "canon" (n=500)
SEED   = 0          # random seed (0, 1, or 2 for reproduction)

In [ ]:
# ── Resolve config from DOMAIN ──
CFG = {
    "trivia": dict(
        K_gen=4, K_train=3, max_tokens=256, batch_q=4,
        min_tokens_feat=3, dataset_name="trivia_qa",
        K_eval_canonical=4,
        k_eval_range=[2, 3, 4],
    ),
    "math": dict(
        K_gen=6, K_train=6, max_tokens=1024, batch_q=2,
        min_tokens_feat=10, dataset_name="hendrycks_math",
        # Canonical K_eval is 4 (even though we also report the full K-sweep).
        K_eval_canonical=4,
        k_eval_range=[2, 3, 4, 5, 6],
    ),
}[DOMAIN]

N = 50 if MODE == "sanity" else 500
PROJ_DIM = 256
TEMP = 0.3
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print(f"Domain: {DOMAIN}  |  Mode: {MODE} (n={N})  |  Seed: {SEED}")
print(f"K_gen={CFG['K_gen']}  K_train={CFG['K_train']}  "
      f"max_tokens={CFG['max_tokens']}  batch_q={CFG['batch_q']}")

In [ ]:
# ── Install dependencies ──
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'accelerate', 'datasets', 'scikit-learn',
    'huggingface_hub', 'scipy'])
print('Dependencies installed.')

In [ ]:
import os, sys, time, json, re, random, datetime
import numpy as np
from collections import Counter

# Import shared utilities
# (adjust path if running from a different directory)
for p in ['.', '..', '/content/paper2_release', '/content/repo/papers/Latent_control']:
    if os.path.exists(os.path.join(p, 'paper2_utils.py')):
        sys.path.insert(0, p)
        break

from paper2_utils import (
    stable_random_projection, resolve_layer_ids, get_layer_indices,
    trajectory_features, train_pairwise_probe, evaluate_at_k,
    print_eval_table, print_stats,
    # Domain-specific
    normalize_answer_text, trivia_is_correct, extract_final_answer,
    get_trivia_aliases, extract_boxed, normalize_math, check_correct_math,
)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def ts():
    return datetime.datetime.now().strftime('[%H:%M:%S]')

## Load Model & Dataset

In [ ]:
# ── Model ──
print(f'{ts()} Loading {MODEL_NAME}...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.bfloat16 if device == 'cuda' else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch_dtype, device_map='auto')
model.eval()

hidden_size = model.config.hidden_size
N_LAYERS = model.config.num_hidden_layers
SPREAD8 = resolve_layer_ids('spread8', N_LAYERS)
proj = stable_random_projection(hidden_size, PROJ_DIM, seed=SEED)
FEAT_IDX = get_layer_indices(SPREAD8, N_LAYERS, PROJ_DIM)
feat_dim_all = PROJ_DIM * N_LAYERS

stop_ids = [tokenizer.eos_token_id]
eot_id = tokenizer.convert_tokens_to_ids('<|eot_id|>')
if isinstance(eot_id, int) and eot_id != tokenizer.unk_token_id:
    stop_ids.append(eot_id)

print(f'{ts()} Model loaded. Layers={N_LAYERS}, spread8={SPREAD8}')
print(f'  Feature dim: {len(FEAT_IDX)} (spread8 × mean_std_last)')

In [ ]:
# ── Dataset ──
from datasets import load_dataset, concatenate_datasets

print(f'{ts()} Loading dataset: {DOMAIN}...')
if DOMAIN == "trivia":
    ds = load_dataset('trivia_qa', 'rc.nocontext', split='validation')
else:  # math
    subs = ['algebra', 'counting_and_probability', 'geometry',
            'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus']
    ds = load_dataset('EleutherAI/hendrycks_math', subs[0], split='test')
    for s in subs[1:]:
        ds = concatenate_datasets([ds,
            load_dataset('EleutherAI/hendrycks_math', s, split='test')])

idxs = list(range(len(ds)))
random.Random(SEED).shuffle(idxs)
all_samples = [ds[i] for i in idxs[:2 * N + 50]]
train_samples = all_samples[:N]
test_samples = all_samples[N:2*N]
print(f'{ts()} Train={N}, Test={N}, Total available={len(ds)}')

## Generate Data + Extract Hidden States

In [ ]:
# ── Hidden state extraction ──
def extract_hs_batch(output_ids, input_len):
    '''Forward pass to extract projected hidden states.'''
    n_seq = output_ids.shape[0]
    attn = (output_ids != tokenizer.pad_token_id).long().to(device)
    with torch.inference_mode():
        out = model(output_ids, attention_mask=attn, output_hidden_states=True)
    results = []
    for ki in range(n_seq):
        gen_mask = output_ids[ki, input_len:] != tokenizer.pad_token_id
        n_gen = gen_mask.sum().item()
        if n_gen < 1:
            results.append(np.zeros((1, feat_dim_all), dtype=np.float32))
            continue
        hs = np.zeros((n_gen, feat_dim_all), dtype=np.float32)
        for i in range(N_LAYERS):
            h = out.hidden_states[i + 1][ki, input_len:input_len + n_gen, :]
            hs[:, i*PROJ_DIM:(i+1)*PROJ_DIM] = h.float().cpu().numpy() @ proj
        results.append(hs)
    del out
    torch.cuda.empty_cache()
    return results

def make_prompt(sample):
    '''Create prompt for the given domain.'''
    if DOMAIN == "trivia":
        msg = ('Answer the following trivia question. Think briefly, '
               'then write your final answer after FINAL:\n\n'
               f'Question: {sample["question"]}')
    else:  # math
        msg = (f'Solve the following math problem step by step. '
               f'Put your final answer in \\boxed{{}}.\n\n'
               f'Problem: {sample["problem"]}')
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': msg}],
        tokenize=False, add_generation_prompt=True)

def check_answer(pred_text, sample):
    '''Check correctness for the given domain.'''
    if DOMAIN == "trivia":
        pred = extract_final_answer(pred_text)
        aliases = get_trivia_aliases(sample['answer'])
        return pred, trivia_is_correct(pred, aliases)
    else:  # math
        pred = extract_boxed(pred_text)
        gold = extract_boxed(sample['solution'])
        return normalize_math(pred), check_correct_math(pred, gold)

# ── Generation loop ──
K = CFG['K_gen']
BATCH_Q = CFG['batch_q']
MAX_TOK = CFG['max_tokens']
MIN_TOK = CFG['min_tokens_feat']

def generate_data(samples, seed_offset=0):
    data = []
    n_correct = n_total = total_tok = 0
    t0 = time.time()
    print(f'{ts()} Generating {len(samples)}q × K={K}...')
    for batch_start in range(0, len(samples), BATCH_Q):
        batch_end = min(batch_start + BATCH_Q, len(samples))
        batch = samples[batch_start:batch_end]
        bs = len(batch)

        prompts = [make_prompt(s) for s in batch]
        enc = tokenizer(prompts, return_tensors='pt', padding=True)
        input_ids = enc['input_ids'].to(device)
        attn_mask = enc['attention_mask'].to(device)
        padded_len = input_ids.shape[1]

        torch.manual_seed(SEED + seed_offset + batch_start)
        if device == 'cuda':
            torch.cuda.manual_seed(SEED + seed_offset + batch_start)

        with torch.inference_mode():
            out_ids = model.generate(
                input_ids, attention_mask=attn_mask,
                max_new_tokens=MAX_TOK, do_sample=True,
                temperature=TEMP, top_k=50,
                num_return_sequences=K, eos_token_id=stop_ids,
                pad_token_id=tokenizer.pad_token_id)

        for bi in range(bs):
            actual_len = attn_mask[bi].sum().item()
            actual_prompt = input_ids[bi, padded_len - actual_len:]
            full_seqs, gen_texts = [], []
            for ki in range(K):
                gen = out_ids[bi * K + ki, padded_len:]
                gen = gen[gen != tokenizer.pad_token_id]
                full_seqs.append(torch.cat([actual_prompt, gen]))
                gen_texts.append(tokenizer.decode(gen, skip_special_tokens=True))
                total_tok += len(gen)

            max_slen = max(s.shape[0] for s in full_seqs)
            padded = torch.full((K, max_slen), tokenizer.pad_token_id,
                                dtype=torch.long, device=device)
            for ki, seq in enumerate(full_seqs):
                padded[ki, :seq.shape[0]] = seq
            hs_list = extract_hs_batch(padded, actual_len)

            attempts = []
            for ki in range(K):
                pred, correct = check_answer(gen_texts[ki], batch[bi])
                feat = trajectory_features(hs_list[ki], min_tokens=MIN_TOK)
                n_tok = len(full_seqs[ki]) - actual_len
                n_total += 1
                n_correct += int(correct)
                attempts.append({
                    'pred': pred, 'correct': correct,
                    'n_tok': n_tok, 'features': feat,
                    'feat_valid': feat is not None,
                })
            data.append({'attempts': attempts})

        done = batch_end
        if done % 50 == 0 or done >= len(samples):
            el = time.time() - t0
            eta = el / done * (len(samples) - done) / 60
            print(f'  {ts()} {done}/{len(samples)} '
                  f'acc={n_correct/max(n_total,1):.0%} '
                  f'{total_tok/max(el,1):.0f}tok/s '
                  f'~{eta:.0f}m left')
    print(f'  Done: {time.time()-t0:.0f}s, acc={n_correct/n_total:.1%}')
    return data

In [ ]:
# ── Generate train + test ──
print('='*60)
print('TRAIN DATA')
print('='*60)
train_data = generate_data(train_samples, seed_offset=0)

print()
print('='*60)
print('TEST DATA')
print('='*60)
test_data = generate_data(test_samples, seed_offset=10000)

## Train Probe & Evaluate

In [ ]:
# ── Train pairwise probe ──
print(f'\nTraining pairwise probe (K_train={CFG["K_train"]})...')
probe, best_C, best_auc, n_pairs = train_pairwise_probe(
    train_data, FEAT_IDX, CFG['K_train'], seed=SEED)
print(f'  Pairs: {n_pairs}, Best C={best_C}, CV AUC={best_auc:.4f}')

# ── Evaluate at multiple K_eval ──
results_by_k = {}
for ke in CFG['k_eval_range']:
    r = evaluate_at_k(test_data, probe, FEAT_IDX, ke, domain=DOMAIN)
    results_by_k[ke] = r

print_eval_table(results_by_k, DOMAIN, SEED, CFG['K_gen'], CFG['K_train'])

# ── Statistical tests (canonical K_eval) ──
canon_k = CFG.get('K_eval_canonical', CFG['k_eval_range'][-1])
canon = results_by_k[canon_k]
print_stats(canon['test_ckpt'], label=f'{DOMAIN} K_eval={canon_k}')

---

## Done!

Compare the output above with the expected values in the header.
For canonical reproduction, set `MODE = "canon"` and run seeds 0, 1, 2.